# Fine-tune an encoder for toxic code review detection (5-fold cross-validation)

One notebook for every encoder model. Edit **only the configuration cell** below, then run all cells.

What it does, for the model in the configuration:
1. **Learning-rate selection** on fold 0: the training folds are split into 90% training and a stratified
   10% validation part (seed 42). One model is trained per learning rate in `LR_GRID`; the learning rate
   with the best validation toxic-class F1 is chosen. The test fold is never used for this choice.
2. **Five folds** with the chosen learning rate. For each fold: train on the other four folds (minus the
   10% validation part) for up to `MAX_EPOCHS` epochs, keep the epoch with the best validation F1, and
   predict the held-out fold.
3. **Outputs** (in `/kaggle/working`): `preds/<RUN_NAME>_<setting>.csv` with one row per comment, written
   after every fold so a crash keeps finished folds, and `logs/<RUN_NAME>_<setting>.json` with the
   chosen learning rate, validation scores, best epochs, run times and library versions.
   Model weights are never saved to disk: the best epoch's weights are only kept in memory.

Test-fold metrics are deliberately **not** computed here; they are computed later from the predictions file.

`SMOKE_TEST = True` runs a quick check (500 training rows, 1 epoch, fold 0 only, first learning rate) and
writes its outputs under `<RUN_NAME>_SMOKE`, so they can never be mixed up with a real run.

In [ ]:
# ======================= CONFIGURATION: the only cell you need to edit =======================
MODEL_ID = "FacebookAI/roberta-base"   # HuggingFace model id
RUN_NAME = "roberta_base"              # name used in output files and as the system name
FOLDS_FILE = "clean"                   # "clean" or "leaky"
LR_GRID = [2e-5, 3e-5, 5e-5]           # large models: [1e-5, 2e-5, 3e-5]; a single value skips the LR search
BATCH_SIZE = 32                        # large models: 16
GRAD_ACCUM = 1                         # large models: 2 (16 x 2 = effective batch of 32)
FP16 = True                            # set to False if the training loss becomes NaN
MAX_LEN = 256                          # tokens; longer comments are truncated
MAX_EPOCHS = 4                         # the best epoch by validation F1 is kept
SMOKE_TEST = True                      # True = quick check: 500 training rows, 1 epoch, fold 0, first LR
FOLDS_TO_RUN = [0, 1, 2, 3, 4]         # only change this to re-run missing folds after a crash
TRUST_REMOTE_CODE = False              # True only for models that need it (NeoBERT)
# ===============================================================================================

In [ ]:
# Must run before torch is imported: use only the first GPU (Kaggle's "T4 x2" has two; using both would
# silently switch to multi-GPU training with a different effective batch size).
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# If the version check in the next cell fails, uncomment this line, run it, then restart the session:
# !pip install -q -U "transformers>=4.48"

In [ ]:
import gc
import glob
import json
import math
import platform
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from packaging.version import Version
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

SEED = 42
EXPECTED_ROWS = {"clean": 12722, "leaky": 19647}
ON_KAGGLE = Path("/kaggle/input").exists()
TRANSFORMERS_V5 = Version(transformers.__version__) >= Version("5.0")

if Version(transformers.__version__) < Version("4.48"):
    raise RuntimeError(f"transformers {transformers.__version__} is too old (ModernBERT needs 4.48 or newer). "
                       "Uncomment the pip line in the cell above, run it, and restart the session.")
assert FOLDS_FILE in EXPECTED_ROWS, "FOLDS_FILE must be 'clean' or 'leaky'"

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
elif ON_KAGGLE:
    raise RuntimeError("No GPU found. In the right-hand panel: Session options -> Accelerator -> GPU T4.")
else:
    GPU_NAME = "none (CPU)"
    print("No GPU: running on CPU (only sensible for a local smoke test)")
USE_FP16 = FP16 and torch.cuda.is_available()

VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "scikit-learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "gpu": GPU_NAME,
}
for name, version in VERSIONS.items():
    print(f"{name:13s} {version}")

In [ ]:
# Locate the fold file. On Kaggle it is in the attached dataset under /kaggle/input/ (the exact folder name
# depends on the dataset name, so it is searched for). Locally, the repository's data/ folder is used.
FOLDS_FILENAME = f"{FOLDS_FILE}_folds.csv"
if ON_KAGGLE:
    candidates = sorted(glob.glob(f"/kaggle/input/**/{FOLDS_FILENAME}", recursive=True))
    OUT_DIR = Path("/kaggle/working")
else:
    candidates = [str(p) for p in (Path("../data") / FOLDS_FILENAME, Path("data") / FOLDS_FILENAME) if p.exists()]
    OUT_DIR = Path("local_output")
if not candidates:
    raise FileNotFoundError(f"{FOLDS_FILENAME} not found. On Kaggle: add your dataset with the fold files "
                            "via 'Add Input' in the right-hand panel.")
FOLDS_PATH = candidates[0]

RUN = RUN_NAME + ("_SMOKE" if SMOKE_TEST else "")
PREDS_PATH = OUT_DIR / "preds" / f"{RUN}_{FOLDS_FILE}.csv"
LOG_PATH = OUT_DIR / "logs" / f"{RUN}_{FOLDS_FILE}.json"
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
PREDS_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(FOLDS_PATH).reset_index(drop=True)
assert {"id", "message", "is_toxic", "fold"} <= set(data.columns), "unexpected columns in the fold file"
assert len(data) == EXPECTED_ROWS[FOLDS_FILE], (
    f"{FOLDS_FILENAME} has {len(data)} rows, expected {EXPECTED_ROWS[FOLDS_FILE]}: wrong or modified file?")
assert data["id"].is_unique and set(data["fold"]) == {0, 1, 2, 3, 4}
data["message"] = data["message"].astype(str)

EPOCHS = 1 if SMOKE_TEST else MAX_EPOCHS
GRID = LR_GRID[:1] if SMOKE_TEST else list(LR_GRID)
FOLDS = [0] if SMOKE_TEST else list(FOLDS_TO_RUN)
print(f"\nFolds file: {FOLDS_PATH} ({len(data):,} rows, {data['is_toxic'].sum():,} toxic)")
print(f"Run: {RUN} | model {MODEL_ID} | LR grid {GRID} | {EPOCHS} epoch(s) | folds {FOLDS} | fp16 {USE_FP16}")
print(f"Outputs: {PREDS_PATH}\n         {LOG_PATH}")

In [ ]:
# Model-specific settings. ModernBERT and Ettin (which uses the ModernBERT architecture) are loaded with
# PyTorch's built-in attention (sdpa), because flash attention does not run on the T4. Older transformers
# versions also torch.compile parts of ModernBERT by default, which can be slow or fail with
# variable-length batches, so that is switched off; transformers 5 removed this option.
model_config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
IS_MODERNBERT = model_config.model_type == "modernbert"
MODEL_KWARGS = {}
if IS_MODERNBERT:
    MODEL_KWARGS["attn_implementation"] = "sdpa"
    if not TRANSFORMERS_V5:
        MODEL_KWARGS["reference_compile"] = False
print(f"Architecture: {model_config.model_type} | extra loading options: {MODEL_KWARGS or 'none'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)

# Tokenize every comment once (no padding here; each batch is padded to its longest comment).
# Only the inputs the model expects are kept (ModernBERT, for example, takes no token_type_ids).
encodings = tokenizer(list(data["message"]), truncation=True, max_length=MAX_LEN)
INPUT_KEYS = [key for key in encodings.keys() if key in tokenizer.model_input_names]
lengths = np.array([len(ids) for ids in encodings["input_ids"]])
TRUNCATED_SHARE = float((lengths >= MAX_LEN).mean())
print(f"Tokens per comment: median {int(np.median(lengths))}, 99th percentile {int(np.percentile(lengths, 99))}; "
      f"truncated at {MAX_LEN}: {TRUNCATED_SHARE:.2%} | model inputs: {INPUT_KEYS}")


class EncodedComments(torch.utils.data.Dataset):
    """The tokenized comments at the given row positions of `data`, with their labels."""

    def __init__(self, rows):
        self.rows = list(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        item = {key: encodings[key][row] for key in INPUT_KEYS}
        item["labels"] = int(data.at[row, "is_toxic"])
        return item

In [ ]:
def toxic_probability(logits):
    """Softmax over the two logits; column 1 is the toxic class."""
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp[:, 1] / exp.sum(axis=1)


def compute_metrics(eval_prediction):
    """Validation metrics for the toxic class; pred = toxic if probability >= 0.5 (same rule as the
    predictions file)."""
    prob = toxic_probability(eval_prediction.predictions)
    pred = (prob >= 0.5).astype(int)
    labels = eval_prediction.label_ids
    return {
        "f1": f1_score(labels, pred, zero_division=0),
        "precision": precision_score(labels, pred, zero_division=0),
        "recall": recall_score(labels, pred, zero_division=0),
    }


def split_fold(fold):
    """Training rows (minus a stratified 10% validation part), validation rows and test rows of a fold."""
    train_all = data[data["fold"] != fold]
    train, val = train_test_split(train_all, test_size=0.1, stratify=train_all["is_toxic"], random_state=SEED)
    if SMOKE_TEST:
        train, _ = train_test_split(train, train_size=500, stratify=train["is_toxic"], random_state=SEED)
    test = data[data["fold"] == fold]
    return train.index, val.index, test.index


class KeepBestEpoch(TrainerCallback):
    """Keeps an in-memory copy of the weights from the epoch with the best validation F1 (the first one if
    several epochs tie). This replaces the Trainer's own save-and-reload of the best checkpoint, which with
    transformers 5 silently skips BERT's LayerNorm weights (they are saved under different names)."""

    def __init__(self):
        self.best_f1 = -1.0
        self.best_state = None

    def on_evaluate(self, args, state, control, model=None, metrics=None, **kwargs):
        if metrics["eval_f1"] > self.best_f1:
            self.best_f1 = metrics["eval_f1"]
            self.best_state = {name: tensor.detach().to("cpu", copy=True)
                               for name, tensor in model.state_dict().items()}


def train_and_predict(fold, lr):
    """Train on one fold's training rows with one learning rate, keep the epoch with the best validation
    F1, and return its validation scores and the toxic probabilities for the fold's test rows."""
    train_rows, val_rows, test_rows = split_fold(fold)
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=2, trust_remote_code=TRUST_REMOTE_CODE, **MODEL_KWARGS)
    # Keep the weights in fp32 (fp16 is only used for the calculations). transformers 5 loads a model in the
    # precision it was saved in, and DeBERTa-v3 is saved in fp16. No change for models saved in fp32.
    model = model.float()

    # Linear learning-rate decay with 10% warmup. The warmup is given as a number of steps, which means the
    # same in transformers 4 and 5 (version 5 removed the warmup_ratio argument).
    updates_per_epoch = math.ceil(math.ceil(len(train_rows) / BATCH_SIZE) / GRAD_ACCUM)
    warmup_steps = int(round(0.1 * updates_per_epoch * EPOCHS))
    output_dir = CHECKPOINT_DIR / f"fold{fold}_lr{lr:g}"

    args = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=lr,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_steps=warmup_steps,
        eval_strategy="epoch",
        save_strategy="no",                 # the best epoch is kept in memory by KeepBestEpoch instead
        fp16=USE_FP16,
        logging_strategy="steps",
        logging_steps=50,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=2 if ON_KAGGLE else 0,
    )
    keep_best = KeepBestEpoch()
    trainer = Trainer(
        model=model,
        callbacks=[keep_best],
        args=args,
        train_dataset=EncodedComments(train_rows),
        eval_dataset=EncodedComments(val_rows),
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8 if USE_FP16 else None),
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    train_seconds = time.time() - start

    history = trainer.state.log_history
    # "loss" = logged every 50 steps; "train_loss" = average over the whole run (always present)
    losses = [entry[key] for entry in history for key in ("loss", "train_loss") if key in entry]
    if any(not math.isfinite(loss) for loss in losses):
        raise RuntimeError("Training loss became NaN/inf. Set FP16 = False in the configuration and run again.")
    average_train_loss = next(entry["train_loss"] for entry in reversed(history) if "train_loss" in entry)
    val_by_epoch = [{"epoch": int(round(entry["epoch"])), "f1": entry["eval_f1"],
                     "precision": entry["eval_precision"], "recall": entry["eval_recall"],
                     "loss": entry["eval_loss"]}
                    for entry in history if "eval_f1" in entry]
    best = max(val_by_epoch, key=lambda e: e["f1"])     # first epoch with the highest F1, as KeepBestEpoch keeps
    if best["f1"] == 0 and SMOKE_TEST:
        print("Note: validation F1 is 0 (no toxic predictions yet). This is normal in the smoke test.")
    elif best["f1"] == 0:
        print("WARNING: validation F1 is 0 in every epoch (the model predicts no toxic comments).")

    # Restore the best epoch's weights. strict=True: every weight must match by name, nothing is skipped.
    trainer.model.load_state_dict(keep_best.best_state, strict=True)
    # Check it: evaluating the validation rows again must give that epoch's F1 and (up to rounding) its loss.
    # The loss changes with any change in the weights, so it also catches partly restored models.
    reloaded = trainer.evaluate()
    best_model_loaded = (abs(reloaded["eval_f1"] - best["f1"]) < 1e-9
                         and abs(reloaded["eval_loss"] - best["loss"]) < 1e-4 * max(1.0, best["loss"]))
    if not best_model_loaded:
        print(f"WARNING: after restoring epoch {best['epoch']}, validation F1/loss are "
              f"{reloaded['eval_f1']:.4f}/{reloaded['eval_loss']:.5f}, but that epoch had "
              f"{best['f1']:.4f}/{best['loss']:.5f}; the best epoch's weights may not have been restored.")
    test_prob = toxic_probability(trainer.predict(EncodedComments(test_rows)).predictions)

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(output_dir, ignore_errors=True)

    print(f"  fold {fold}, lr {lr:g}: best epoch {best['epoch']} (validation F1 {best['f1']:.4f}), "
          f"{train_seconds / 60:.1f} min, average training loss {average_train_loss:.4f}")
    return {
        "lr": lr,
        "best_epoch": best["epoch"],
        "best_val_f1": best["f1"],
        "val_by_epoch": val_by_epoch,
        "average_train_loss": average_train_loss,
        "best_model_loaded": best_model_loaded,
        "train_rows": len(train_rows),
        "val_rows": len(val_rows),
        "train_seconds": round(train_seconds, 1),
        "test_rows": list(test_rows),
        "test_prob": test_prob,
    }


def save_fold_predictions(fold, result):
    """Append one fold's test predictions to the predictions file (header only for the first fold)."""
    test = data.loc[result["test_rows"]]
    prob = result["test_prob"]
    fold_preds = pd.DataFrame({
        "id": test["id"].to_numpy(),
        "fold": fold,
        "label": test["is_toxic"].to_numpy(),
        "prob_toxic": prob,
        "pred": (prob >= 0.5).astype(int),
        "system": RUN,
        "setting": FOLDS_FILE,
    })
    fold_preds.to_csv(PREDS_PATH, mode="a", header=not PREDS_PATH.exists(), index=False)


def summary(result):
    """The parts of a training result that go into the JSON log."""
    return {key: value for key, value in result.items() if key not in ("test_rows", "test_prob")}


def save_log(log):
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")

In [ ]:
# Resume support for interactive sessions: folds already in the predictions file are skipped, and a chosen
# learning rate in an existing log for the same configuration is reused.
done_folds = set(pd.read_csv(PREDS_PATH)["fold"]) if PREDS_PATH.exists() else set()
log = {
    "run": RUN, "model_id": MODEL_ID, "folds_file": FOLDS_FILE, "folds_path": FOLDS_PATH,
    "config": {"lr_grid": GRID, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "fp16": USE_FP16,
               "max_len": MAX_LEN, "max_epochs": EPOCHS, "smoke_test": SMOKE_TEST, "seed": SEED,
               "weight_decay": 0.01, "warmup": "10% of steps, linear decay", "validation": "stratified 10%",
               "model_kwargs": {k: str(v) for k, v in MODEL_KWARGS.items()}},
    "versions": VERSIONS,
    "truncated_share": TRUNCATED_SHARE,
    "started": datetime.now().isoformat(timespec="seconds"),
    "lr_search": {}, "chosen_lr": None, "folds": {},
}
if LOG_PATH.exists():
    previous = json.loads(LOG_PATH.read_text(encoding="utf-8"))
    if previous.get("config", {}).get("lr_grid") == GRID and previous.get("model_id") == MODEL_ID:
        log["lr_search"], log["chosen_lr"] = previous.get("lr_search", {}), previous.get("chosen_lr")
        log["folds"] = {k: v for k, v in previous.get("folds", {}).items() if int(k) in done_folds}
if done_folds:
    print(f"Resuming: folds {sorted(done_folds)} are already in {PREDS_PATH} and will be skipped.")

In [ ]:
# Step 1: learning-rate selection on fold 0 (validation F1 only).
lr_search_results = {}
if log["chosen_lr"] is not None:
    print(f"Learning rate {log['chosen_lr']:g} reused from the existing log.")
elif len(GRID) == 1:
    log["chosen_lr"] = GRID[0]
    print(f"Single learning rate {GRID[0]:g}: no search needed.")
else:
    print(f"Learning-rate search on fold 0: {GRID}")
    for lr in GRID:
        lr_search_results[lr] = train_and_predict(0, lr)
        log["lr_search"][f"{lr:g}"] = summary(lr_search_results[lr])
        save_log(log)
    log["chosen_lr"] = max(GRID, key=lambda lr: lr_search_results[lr]["best_val_f1"])
    print("Validation F1 per learning rate: "
          + ", ".join(f"{lr:g}: {lr_search_results[lr]['best_val_f1']:.4f}" for lr in GRID)
          + f"  ->  chosen {log['chosen_lr']:g}")
save_log(log)
CHOSEN_LR = log["chosen_lr"]

In [ ]:
# Step 2: all folds with the chosen learning rate. Fold 0 with the chosen learning rate was already trained
# during the search (same data split, same seed), so its test predictions are reused instead of training it
# again; they played no part in choosing the learning rate.
for fold in FOLDS:
    if fold in done_folds:
        continue
    if fold == 0 and CHOSEN_LR in lr_search_results:
        result = lr_search_results[CHOSEN_LR]
        print(f"  fold 0: reusing the lr {CHOSEN_LR:g} model from the learning-rate search")
    else:
        result = train_and_predict(fold, CHOSEN_LR)
    save_fold_predictions(fold, result)
    log["folds"][str(fold)] = summary(result)
    log["finished"] = datetime.now().isoformat(timespec="seconds")
    save_log(log)

In [ ]:
# Final check of the outputs (no test metrics here: those are computed from the predictions file later).
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
preds = pd.read_csv(PREDS_PATH)
expected_folds = set(FOLDS) | done_folds
assert preds["id"].is_unique, "duplicate ids in the predictions file"
assert set(preds["fold"]) == expected_folds, f"folds in file {sorted(set(preds['fold']))} != {sorted(expected_folds)}"
assert len(preds) == data["fold"].isin(expected_folds).sum(), "row count does not match the folds"
assert list(preds.columns) == ["id", "fold", "label", "prob_toxic", "pred", "system", "setting"]

print(f"Chosen learning rate: {CHOSEN_LR:g}")
for fold, info in sorted(log["folds"].items()):
    print(f"  fold {fold}: best epoch {info['best_epoch']}, validation F1 {info['best_val_f1']:.4f}, "
          f"{info['train_seconds'] / 60:.1f} min")
print(f"\nPredictions: {PREDS_PATH} ({len(preds):,} rows, folds {sorted(set(preds['fold']))})")
print(f"Log:         {LOG_PATH}")
if SMOKE_TEST:
    print("\nSMOKE TEST finished. For the real run set SMOKE_TEST = False.")